# White-box testing — Conversor de bases numéricas

Leemos el código de `src/func/conversions.py` y `src/func/verify.py` y diseñamos casos para recorrer statements, branches, loops y paths. Cada caso indica el RF que apoya.

## Setup

Importamos los módulos reales de la app desde `src`.

In [ ]:
import os
import sys
import importlib
import unittest

SRC = os.path.abspath(os.path.join(os.getcwd(), '..', 'src'))
if SRC not in sys.path:
    sys.path.insert(0, SRC)

import func.verify
import func.conversions
importlib.reload(func.verify)
importlib.reload(func.conversions)
from func.verify import verify
from func.conversions import conversions

print('src:', SRC)


src: /home/jandro/programming/modul/src
True


In [18]:
def _run(class_name):
    suite = unittest.defaultTestLoader.loadTestsFromTestCase(globals()[class_name])
    result = unittest.TextTestRunner(verbosity=2).run(suite)
    assert result.wasSuccessful()
    return result

## Código bajo prueba

`verifyNumType` — una decisión D1: si `int(cleanNum, base)` lanza `ValueError`.

```python
def verifyNumType(n, base):
    try:
        cleanNum = n.strip()      # S1
        int(cleanNum, base)       # S2
        return True               # S3
    except ValueError:
        return False              # S4
```

`verifyNoNegOrDec` — decisión compuesta D1 con dos condiciones, C1 `'-' in cleanNum` y C2 `'.' in cleanNum`.

```python
def verifyNoNegOrDec(n):
    cleanNum = n.strip()
    if ('-' in cleanNum or '.' in cleanNum):    # D1
        return False                            # S2
    else:
        return True                             # S3
```

`toDecimal` y `decimalTo` — cada una con D1 (atajo del cero) y D2 (loop).

```python
def toDecimal(n, base):
    if n == '0': return (0, [], [])     # D1
    for i in range(len(numberList)):    # D2
        ...

def decimalTo(n, base):
    if n == 0: return ('0', [], [])     # D1
    while up > 0:                       # D2
        ...
```

## 1. `verifyNumType` — statement y branch coverage (RF-05, RF-06, RF-07)

WB-01 y WB-02 cubren las dos ramas de D1 (100% statement + branch). WB-03 y WB-04 van por la rama true y confirman RF-06 y RF-07.

| ID | Entrada | base | Branch | Esperado |
|---|---|---|---|---|
| WB-01 | `1010` | 2 | D1 true | True |
| WB-02 | `2` | 2 | D1 false | False |

In [19]:
class VerifyCajaBlanca(unittest.TestCase):

    def test_wb01_rama_verdadera(self):
        """WB-01 (D1 true): un número analizable devuelve True"""
        self.assertTrue(verify.verifyNumType('1010', 2))

    def test_wb02_rama_falsa(self):
        """WB-02 (D1 false): un número no analizable devuelve False"""
        self.assertFalse(verify.verifyNumType('2', 2))

    def test_wb03_rama_verdadera_hex_minuscula(self):
        """WB-03 (D1 true, RF-06): el hex en minúscula también se analiza"""
        self.assertTrue(verify.verifyNumType('abc', 16))

    def test_wb04_rama_verdadera_ceros(self):
        """WB-04 (D1 true, RF-07): los ceros a la izquierda se analizan"""
        self.assertTrue(verify.verifyNumType('0077', 8))


_run('VerifyCajaBlanca')

test_wb01_rama_verdadera (__main__.VerifyCajaBlanca.test_wb01_rama_verdadera)
WB-01 (D1 true): un número analizable devuelve True ... ok
test_wb02_rama_falsa (__main__.VerifyCajaBlanca.test_wb02_rama_falsa)
WB-02 (D1 false): un número no analizable devuelve False ... ok
test_wb03_rama_verdadera_hex_minuscula (__main__.VerifyCajaBlanca.test_wb03_rama_verdadera_hex_minuscula)
WB-03 (D1 true, RF-06): el hex en minúscula también se analiza ... ok
test_wb04_rama_verdadera_ceros (__main__.VerifyCajaBlanca.test_wb04_rama_verdadera_ceros)
WB-04 (D1 true, RF-07): los ceros a la izquierda se analizan ... ok

----------------------------------------------------------------------
Ran 4 tests in 0.002s

OK


<unittest.runner.TextTestResult run=4 errors=0 failures=0>

## 2. `verifyNoNegOrDec` — condición compuesta (solo enteros no negativos)

D1 es un `or`, con short-circuit: si C1 es true, C2 no se evalúa. Para cubrir ambas ramas y ambas condiciones:

| ID | Entrada | C1 (`-`) | C2 (`.`) | Branch | Esperado |
|---|---|---|---|---|---|
| WB-01 | `10` | false | false | false | True |
| WB-02 | `-5` | true | — | true por C1 | False |
| WB-03 | `1.23` | false | true | true por C2 | False |

In [20]:
class NoNegOrDecCajaBlanca(unittest.TestCase):

    def test_wb01_rama_falsa(self):
        """WB-01 (D1 false): '10' devuelve True"""
        self.assertTrue(verify.verifyNoNegOrDec('10'))

    def test_wb02_verdadera_por_signo(self):
        """WB-02 (D1 true por C1): '-5' devuelve False"""
        self.assertFalse(verify.verifyNoNegOrDec('-5'))

    def test_wb03_verdadera_por_punto(self):
        """WB-03 (D1 true por C2): '1.23' devuelve False"""
        self.assertFalse(verify.verifyNoNegOrDec('1.23'))


_run('NoNegOrDecCajaBlanca')

test_wb01_rama_falsa (__main__.NoNegOrDecCajaBlanca.test_wb01_rama_falsa)
WB-01 (D1 false): '10' devuelve True ... ok
test_wb02_verdadera_por_signo (__main__.NoNegOrDecCajaBlanca.test_wb02_verdadera_por_signo)
WB-02 (D1 true por C1): '-5' devuelve False ... ok
test_wb03_verdadera_por_punto (__main__.NoNegOrDecCajaBlanca.test_wb03_verdadera_por_punto)
WB-03 (D1 true por C2): '1.23' devuelve False ... ok

----------------------------------------------------------------------
Ran 3 tests in 0.002s

OK


<unittest.runner.TextTestResult run=3 errors=0 failures=0>

## 3. `toDecimal` — branch y loop coverage (RF-09, RF-15)

D1 atajo del cero, D2 el `for`. Loop testing con cero, una y varias vueltas.

| ID | Entrada | base | D1 | Vueltas | Esperado |
|---|---|---|---|---|---|
| WB-01 | `0` | 16 | true | 0 | 0 |
| WB-02 | `7` | 8 | false | 1 | 7 |
| WB-03 | `FF` | 16 | false | 2 | 255 |
| WB-04 | `1010` | 2 | false | 4 | 10 |

In [21]:
class ToDecimalCajaBlanca(unittest.TestCase):

    def test_wb01_atajo_cero(self):
        """WB-01 (D1 true, RF-15): '0' devuelve (0, [], []) sin loop"""
        self.assertEqual(conversions.toDecimal('0', 16), (0, [], []))

    def test_wb02_un_digito_una_vuelta(self):
        """WB-02 (D1 false, loop x1): '7'(8) == 7"""
        value, steps, _ = conversions.toDecimal('7', 8)
        self.assertEqual(value, 7)
        self.assertEqual(len(steps), 1)

    def test_wb03_dos_vueltas(self):
        """WB-03 (loop x2, RF-09): 'FF'(16) == 255"""
        value, steps, _ = conversions.toDecimal('FF', 16)
        self.assertEqual(value, 255)
        self.assertEqual(len(steps), 2)

    def test_wb04_varias_vueltas(self):
        """WB-04 (loop x4, RF-09): '1010'(2) == 10"""
        value, steps, _ = conversions.toDecimal('1010', 2)
        self.assertEqual(value, 10)
        self.assertEqual(len(steps), 4)


_run('ToDecimalCajaBlanca')

test_wb01_atajo_cero (__main__.ToDecimalCajaBlanca.test_wb01_atajo_cero)
WB-01 (D1 true, RF-15): '0' devuelve (0, [], []) sin loop ... ok
test_wb02_un_digito_una_vuelta (__main__.ToDecimalCajaBlanca.test_wb02_un_digito_una_vuelta)
WB-02 (D1 false, loop x1): '7'(8) == 7 ... ok
test_wb03_dos_vueltas (__main__.ToDecimalCajaBlanca.test_wb03_dos_vueltas)
WB-03 (loop x2, RF-09): 'FF'(16) == 255 ... ok
test_wb04_varias_vueltas (__main__.ToDecimalCajaBlanca.test_wb04_varias_vueltas)
WB-04 (loop x4, RF-09): '1010'(2) == 10 ... ok

----------------------------------------------------------------------
Ran 4 tests in 0.003s

OK


<unittest.runner.TextTestResult run=4 errors=0 failures=0>

## 4. `decimalTo` — branch y loop coverage (RF-10, RF-12, RF-14, RF-15)

D1 atajo del cero, D2 el `while`. WB-04 fuerza la línea de `reverse_conversion` (residuo >= 10 -> letras, RF-12).

| ID | n | base | D1 | Vueltas | Esperado |
|---|---|---|---|---|---|
| WB-01 | 0 | 2 | true | 0 | '0' |
| WB-02 | 1 | 2 | false | 1 | '1' |
| WB-03 | 10 | 2 | false | 4 | '1010' |
| WB-04 | 2748 | 16 | false | 3 | 'ABC' |

In [22]:
class DecimalToCajaBlanca(unittest.TestCase):

    def test_wb01_atajo_cero(self):
        """WB-01 (D1 true, RF-15): 0 devuelve ('0', [], []) sin loop"""
        self.assertEqual(conversions.decimalTo(0, 2), ('0', [], []))

    def test_wb02_una_vuelta(self):
        """WB-02 (loop x1): 1 -> '1'"""
        digits, steps, _ = conversions.decimalTo(1, 2)
        self.assertEqual(digits, '1')
        self.assertEqual(len(steps), 1)

    def test_wb03_varias_vueltas_con_pasos(self):
        """WB-03 (loop x4, RF-14): 10 -> '1010' y 4 pasos"""
        digits, steps, _ = conversions.decimalTo(10, 2)
        self.assertEqual(digits, '1010')
        self.assertEqual(len(steps), 4)
        self.assertEqual(steps[0], '10 / 2 = 5 remainder = 0')

    def test_wb04_mapeo_a_letras(self):
        """WB-04 (línea de mapeo, RF-12): 2748 -> 'ABC'"""
        self.assertEqual(conversions.decimalTo(2748, 16)[0], 'ABC')


_run('DecimalToCajaBlanca')

test_wb01_atajo_cero (__main__.DecimalToCajaBlanca.test_wb01_atajo_cero)
WB-01 (D1 true, RF-15): 0 devuelve ('0', [], []) sin loop ... ok
test_wb02_una_vuelta (__main__.DecimalToCajaBlanca.test_wb02_una_vuelta)
WB-02 (loop x1): 1 -> '1' ... ok
test_wb03_varias_vueltas_con_pasos (__main__.DecimalToCajaBlanca.test_wb03_varias_vueltas_con_pasos)
WB-03 (loop x4, RF-14): 10 -> '1010' y 4 pasos ... ok
test_wb04_mapeo_a_letras (__main__.DecimalToCajaBlanca.test_wb04_mapeo_a_letras)
WB-04 (línea de mapeo, RF-12): 2748 -> 'ABC' ... ok

----------------------------------------------------------------------
Ran 4 tests in 0.003s

OK


<unittest.runner.TextTestResult run=4 errors=0 failures=0>

## 5. Path coverage — flow completo (RF-09, RF-10, RF-11)

Flow de datos: base origen -> `toDecimal` -> decimal -> `decimalTo` por cada base destino. WB-01 recorre el cuerpo de ambas; WB-02 toma la salida temprana de D1 en las dos.

In [23]:
class TuberiaCajaBlanca(unittest.TestCase):

    def test_wb01_camino_no_cero(self):
        """WB-01: FF(16) -> 255 -> binario/octal/hex (corren los dos loops)"""
        decimal = conversions.toDecimal('FF', 16)[0]
        self.assertEqual(decimal, 255)
        self.assertEqual(conversions.decimalTo(decimal, 2)[0], '11111111')
        self.assertEqual(conversions.decimalTo(decimal, 8)[0], '377')
        self.assertEqual(conversions.decimalTo(decimal, 16)[0], 'FF')

    def test_wb02_camino_cero(self):
        """WB-02: '0' toma la salida temprana de D1 en ambas funciones"""
        decimal = conversions.toDecimal('0', 16)[0]
        self.assertEqual(decimal, 0)
        self.assertEqual(conversions.decimalTo(decimal, 2)[0], '0')
        self.assertEqual(conversions.decimalTo(decimal, 16)[0], '0')


_run('TuberiaCajaBlanca')

test_wb01_camino_no_cero (__main__.TuberiaCajaBlanca.test_wb01_camino_no_cero)
WB-01: FF(16) -> 255 -> binario/octal/hex (corren los dos loops) ... ok
test_wb02_camino_cero (__main__.TuberiaCajaBlanca.test_wb02_camino_cero)
WB-02: '0' toma la salida temprana de D1 en ambas funciones ... ok

----------------------------------------------------------------------
Ran 2 tests in 0.002s

OK


<unittest.runner.TextTestResult run=2 errors=0 failures=0>

## 6. Coverage medido con `trace`

Contamos qué líneas de `conversions.py` se ejecutan al ejercitar ambas funciones.

In [24]:
import trace as trace_mod

def _exercise_all():
    conversions.toDecimal('0', 16)
    conversions.toDecimal('1010', 2)
    conversions.toDecimal('FF', 16)
    conversions.decimalTo(0, 2)
    conversions.decimalTo(10, 2)
    conversions.decimalTo(2748, 16)
    verify.verifyNumType('1010', 2)
    verify.verifyNumType('2', 2)

tracer = trace_mod.Trace(count=True, trace=False)
tracer.runfunc(_exercise_all)
results = tracer.results()

conv_file = os.path.join(SRC, 'func', 'conversions.py')
executed = {line for (fname, line) in results.counts if os.path.abspath(fname) == os.path.abspath(conv_file)}
print('Líneas ejecutadas en conversions.py:', sorted(executed))
print('Líneas distintas ejecutadas:', len(executed))

Líneas ejecutadas en conversions.py: [18, 19, 20, 21, 22, 23, 24, 25, 26, 28, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43]
Líneas distintas ejecutadas: 22


## 7. Fuera del alcance estructural

RF-01, RF-02, RF-03, RF-13, RF-16 y RF-17 son I/O del menú en `index.py` (`input()`, `print()`, loop del menú); se validan ejecutando la app. RF-08 es una decisión (`decimal > 1000000`) en `index.calculate`, cubierta con los boundary values del cuaderno de caja negra.

## 8. Batería completa

In [25]:
_clases = [
    'VerifyCajaBlanca',
    'NoNegOrDecCajaBlanca',
    'ToDecimalCajaBlanca',
    'DecimalToCajaBlanca',
    'TuberiaCajaBlanca',
]
_suite = unittest.TestSuite()
for name in _clases:
    _suite.addTests(unittest.defaultTestLoader.loadTestsFromTestCase(globals()[name]))
_result = unittest.TextTestRunner(verbosity=2).run(_suite)
assert _result.wasSuccessful()

test_wb01_rama_verdadera (__main__.VerifyCajaBlanca.test_wb01_rama_verdadera)
WB-01 (D1 true): un número analizable devuelve True ... ok
test_wb02_rama_falsa (__main__.VerifyCajaBlanca.test_wb02_rama_falsa)
WB-02 (D1 false): un número no analizable devuelve False ... ok
test_wb03_rama_verdadera_hex_minuscula (__main__.VerifyCajaBlanca.test_wb03_rama_verdadera_hex_minuscula)
WB-03 (D1 true, RF-06): el hex en minúscula también se analiza ... ok
test_wb04_rama_verdadera_ceros (__main__.VerifyCajaBlanca.test_wb04_rama_verdadera_ceros)
WB-04 (D1 true, RF-07): los ceros a la izquierda se analizan ... ok
test_wb01_rama_falsa (__main__.NoNegOrDecCajaBlanca.test_wb01_rama_falsa)
WB-01 (D1 false): '10' devuelve True ... ok
test_wb02_verdadera_por_signo (__main__.NoNegOrDecCajaBlanca.test_wb02_verdadera_por_signo)
WB-02 (D1 true por C1): '-5' devuelve False ... ok
test_wb03_verdadera_por_punto (__main__.NoNegOrDecCajaBlanca.test_wb03_verdadera_por_punto)
WB-03 (D1 true por C2): '1.23' devuelve Fa